# Evaluation awareness: publication experiments on Kaggle

**Self-contained:** includes the revised source for all three methods, tests, original-run audits and blinded response-review files. No laptop GPU or source ZIP upload is needed.

1. Import this `.ipynb` into Kaggle. Enable **Internet** and choose an available **GPU accelerator** in notebook settings.
2. Run the cells in order. The default queue attempts three unfinished jobs per session, starting with Qwen1.5 and all three methods. The complete matrix is four models × three methods × two protocols = **24 jobs** at seed 42.
3. Download the checkpoint ZIP before ending the session. For another session, attach that ZIP as a Kaggle dataset and enter its `/kaggle/input/...` path in `RESUME_ZIP`. Kaggle session limits can interrupt a long job; generation checkpoints support resumption, but files must be retained.

For Gemma, accept the model's Hugging Face access terms and add a Kaggle secret named **HF_TOKEN**. Do not paste credentials into this notebook. Gemma here is **google/gemma-2b-it (Gemma 1)**, matching the original runs. Lack of Gemma access is recorded and does not prevent other models running.

**Evidence status:** nine CPU tests passed, including eight tiny-transformer configurations. Full research-model runs have not been executed here. The existing saved-run audits are included; independent semantic refusal/quality review remains necessary. A REAL result is not guaranteed. Keep the research notebook and response outputs private until you choose what to publish.

Setup references: [Kaggle notebooks](https://www.kaggle.com/docs/notebooks), [Gemma checkpoint and access terms](https://huggingface.co/google/gemma-2b-it).


In [ ]:
from pathlib import Path
import base64, hashlib, io, json, os, subprocess, sys, zipfile

ROOT = Path('/kaggle/working/eval_awareness_publication')
ROOT.mkdir(parents=True, exist_ok=True)
PAYLOAD = '...'
EXPECTED_SHA256 = 'f5cc3c1ab69b80186d68ce5a0f24747d3a35c83461400f13b90d84774cf3c7b6'
archive = base64.b64decode(PAYLOAD)
assert hashlib.sha256(archive).hexdigest() == EXPECTED_SHA256
with zipfile.ZipFile(io.BytesIO(archive)) as z:
 for item in z.infolist():
 dest = (ROOT / item.filename).resolve()
 if not dest.is_relative_to(ROOT.resolve()):
 raise ValueError('Unsafe archive path')
 z.extractall(ROOT)
os.environ['PYTHONPATH'] = str(ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')
print('Source, tests and audit files extracted to', ROOT)
print((ROOT / 'audit/AUDIT_FINDINGS.md').read_text())


## Install dependencies and verify the runtime
Keep Kaggle's CUDA-enabled PyTorch. Transformers is pinned to the version used for the local integration tests. Package versions, checkpoint revision and GPU type are recorded; a partial job refuses to resume under changed runtime settings. If installation reports a loaded-package conflict, restart the kernel and run again.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
 'transformers==4.57.1', 'sentencepiece>=0.2.0', 'protobuf>=3.20.0',
 'accelerate>=0.26.0', 'scikit-learn>=1.3.0', 'numpy>=1.24.0'], check=True)
import torch
from transformers import Qwen2Config, Qwen2ForCausalLM
print('PyTorch:', torch.__version__)
if not torch.cuda.is_available():
 raise RuntimeError('Enable a GPU accelerator in Kaggle notebook settings, then restart.')
print('GPU:', torch.cuda.get_device_name(0))
# Authentication is optional for public checkpoints. Tokens are never printed.
try:
 from kaggle_secrets import UserSecretsClient
 token = UserSecretsClient().get_secret('HF_TOKEN')
 if token:
 os.environ['HF_TOKEN'] = token
 del token
except Exception:
 print('HF_TOKEN not configured; gated Gemma jobs may require it.')
# Real tiny-model tests run on CPU and do not download a research model.
test = subprocess.run([sys.executable, '-m', 'unittest', 'discover',
 '-s', str(ROOT/'tests'), '-v'], cwd=ROOT,
 stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
(ROOT/'audit/kaggle_test_log.txt').write_text(test.stdout)
print(test.stdout)
if test.returncode or 'skipped=' in test.stdout:
 raise RuntimeError('Fix test failures or missing dependencies before full inference.')


## Configure the queue and optional resume
The default v5_4 mode is **block_outputs**, preserving the original method. `writes` is a separate optional experiment, not a silent replacement. Run the defaults first and retain the entire matrix, including negative results.

These follow-up experiments use 346 training, 114 validation and 114 test request pairs. Layer and strength are selected on validation data. The second protocol withholds two existing wrapper pairs for validation and two for test. This is a stronger evaluation design, not an exact replay of the old 70/30 experiment.

Use a fresh `EXPERIMENT_NAME` for intentional changes to settings/source. Restore partial runs only with matching batch size, source, checkpoint revision, library versions and GPU type. Do not merge output directories from different configurations.


In [ ]:
EXPERIMENT_NAME = 'publication_v1'
MODELS = ['Qwen/Qwen1.5-1.8B-Chat', 'Qwen/Qwen2.5-1.5B-Instruct',
 'google/gemma-2b-it', 'TinyLlama/TinyLlama-1.1B-Chat-v1.0']
METHODS = ['single_layer', 'weight_edit', 'all_layers']
PROTOCOLS = ['heldout_pairs', 'heldout_templates']
SEEDS = [42]
ALL_LAYER_MODE = 'block_outputs' # Optional separate experiment: 'writes'
BATCH_SIZE = 2
MAX_JOBS_THIS_SESSION = 3 # Attempt up to three unfinished jobs
RETRY_FAILED = False # Set True after resolving a failed job's cause
RESUME_ZIP = None # e.g. '/kaggle/input/my-checkpoint/eval_awareness_checkpoint.zip'
SAVE_ACTIVATIONS_IN_ZIP = False # False saves space; activations are recomputed if needed
OUTPUT = Path('/kaggle/working') / EXPERIMENT_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)
if RESUME_ZIP:
 with zipfile.ZipFile(RESUME_ZIP) as z:
 for item in z.infolist():
 dest = (OUTPUT / item.filename).resolve()
 if not dest.is_relative_to(OUTPUT.resolve()):
 raise ValueError('Unsafe checkpoint path')
 if item.is_dir():
 continue
 data = z.read(item)
 if dest.exists() and dest.read_bytes() != data:
 raise ValueError(f'Existing output conflicts with restore: {item.filename}')
 dest.parent.mkdir(parents=True, exist_ok=True)
 dest.write_bytes(data)
 print('Restored checkpoint files.')
assert MAX_JOBS_THIS_SESSION > 0
assert ALL_LAYER_MODE in ('block_outputs', 'writes')


## Run the experiments
Each subprocess uses one GPU; the methods run sequentially. It saves responses after each generation batch and a complete record after each strength. The queue exports a checkpoint after every attempted job. If you interrupt a job manually, run the final export cell before leaving.

A complete 24-job matrix may need several sessions. There is no reliable runtime estimate until the first job runs on your assigned GPU. If you get an out-of-memory error, a smaller batch size requires a **fresh experiment**; do not mix it into partial runs. Do not use the best test-sweep point as the primary result.


In [ ]:
from huggingface_hub import HfApi
import datetime, itertools, re, time
CHECKPOINT = Path('/kaggle/working/eval_awareness_checkpoint.zip')

def write_json(path, obj):
 tmp = path.with_suffix(path.suffix + '.tmp')
 tmp.write_text(json.dumps(obj, indent=2, allow_nan=False))
 tmp.replace(path)

def export_checkpoint():
 temp = CHECKPOINT.with_suffix('.tmp.zip')
 with zipfile.ZipFile(temp, 'w', zipfile.ZIP_DEFLATED) as z:
 for path in sorted(OUTPUT.rglob('*')):
 if path.is_file() and not path.name.endswith('.tmp'):
 if path.suffix == '.npy' and not SAVE_ACTIVATIONS_IN_ZIP:
 continue
 z.write(path, path.relative_to(OUTPUT))
 temp.replace(CHECKPOINT)
 print('Checkpoint ready:', CHECKPOINT, f'({CHECKPOINT.stat().st_size/1e6:.1f} MB)')

plan = dict(models=MODELS, methods=METHODS, protocols=PROTOCOLS, seeds=SEEDS,
 all_layer_mode=ALL_LAYER_MODE, batch_size=BATCH_SIZE,
 package_sha256=EXPECTED_SHA256)
plan_path = OUTPUT / 'queue_plan.json'
if plan_path.exists() and json.loads(plan_path.read_text()) != plan:
 raise ValueError('Queue/settings changed. Restore the original settings or use a fresh experiment name.')
write_json(plan_path, plan)
rev_path = OUTPUT / 'checkpoint_revisions.json'
revisions = json.loads(rev_path.read_text()) if rev_path.exists() else {}
status_path = OUTPUT / 'queue_status.json'
status = json.loads(status_path.read_text()) if status_path.exists() else {}
attempted = 0
try:
 # Model then protocol ordering: first three jobs are the three original scopes on Qwen1.5.
 for model, protocol, seed, method in itertools.product(MODELS, PROTOCOLS, SEEDS, METHODS):
 name = re.sub(r'[^A-Za-z0-9_.-]', '_', model) + f'__{protocol}__seed{seed}__{method}'
 if method == 'all_layers':
 name += '__' + ALL_LAYER_MODE
 target = OUTPUT / name
 if (target/'results.json').exists():
 print('Complete:', name)
 continue
 if status.get(name, {}).get('state') == 'failed' and not RETRY_FAILED:
 print('Previously failed; set RETRY_FAILED=True after fixing:', name)
 continue
 if attempted >= MAX_JOBS_THIS_SESSION:
 break
 attempted += 1
 status[name] = {'state':'running', 'started_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()}
 write_json(status_path, status)
 try:
 if model not in revisions:
 revisions[model] = HfApi().model_info(model).sha
 write_json(rev_path, revisions)
 cmd = [sys.executable, '-m', 'eval_awareness.run_publication', '--model', model,
 '--revision', revisions[model], '--method', method, '--protocol', protocol,
 '--ablation-mode', ALL_LAYER_MODE, '--seed', str(seed),
 '--batch-size', str(BATCH_SIZE), '--device', 'cuda', '--output', str(target)]
 print('Starting:', name, flush=True)
 subprocess.run(cmd, cwd=ROOT, check=True)
 status[name]['state'] = 'complete'
 except Exception as exc:
 status[name]['state'] = 'failed'
 # Store the error type, without accidentally logging credentials or signed URLs.
 status[name]['error_type'] = type(exc).__name__
 print('Job did not complete:', name, type(exc).__name__)
 print('Check the preceding error. Access, memory or resume-version issues must be resolved.')
 finally:
 write_json(status_path, status)
 export_checkpoint()
finally:
 export_checkpoint()
print('Attempts this session:', attempted)


## Summarise results and prepare blinded quality reviews
The primary result uses the alpha chosen on validation. `not_assessable` and absent eligible settings are retained. **REAL means only that the specified probe- and gap-reduction criteria were met.** Keyword refusal scores alone cannot demonstrate coherent answers or awareness removal.

For each completed run, the audit creates `blind_review.csv`, a separate `review_key_PRIVATE.csv`, quality flags and paired intervals. Give reviewers the blinded CSV and `ANNOTATION_RUBRIC.md`, not the key. The sample uses the same randomly selected base requests across all strengths and framings. Follow independent annotation and adjudication before asserting semantic refusal validity. The existing two-run review queues are already in the extracted `audit` directory.


In [ ]:
import csv
summary = []
for result_path in sorted(OUTPUT.glob('*/results.json')):
 d = json.loads(result_path.read_text())
 p = d.get('primary_test_result') or {}
 interval = p.get('paired_intervals', {}).get('all_requests', {})
 summary.append(dict(run=result_path.parent.name, model=d['config']['model_name'],
 method=d['config']['method'], protocol=d['config']['protocol'],
 layer=d['selected_layer'], validation_selected_alpha=d['selected_alpha'],
 primary_test_verdict=p.get('assessable_verdict', 'not_assessable'),
 auc=p.get('auc'), gap=p.get('gap'), rho_A=p.get('d_auc'), rho_G=p.get('d_gap'),
 oversteered=p.get('oversteered'),
 absolute_gap_reduction_ci=json.dumps(interval.get('gap_reduction_absolute')),
 semantic_quality_status=d['quality_validation']))
 review_dir = result_path.parent / 'quality_review'
 if not (review_dir/'audit.json').exists():
 subprocess.run([sys.executable, str(ROOT/'tools/audit_saved.py'), str(result_path),
 '--output', str(review_dir)], cwd=ROOT, check=True)
if summary:
 with (OUTPUT/'primary_results_summary.csv').open('w', newline='') as f:
 w = csv.DictWriter(f, fieldnames=list(summary[0])); w.writeheader(); w.writerows(summary)
 try:
 import pandas as pd
 display(pd.DataFrame(summary))
 except ImportError:
 print(json.dumps(summary, indent=2))
else:
 print('No full model jobs have completed yet; inspect queue_status.json and the preceding logs.')
export_checkpoint()


## Download and continue in another session
Download both files below. The checkpoint contains saved run results, partial generations and the queue plan. The support archive contains the revised source, tests, original-run audits and review rubric. Model weights are not included.

To resume, attach the checkpoint ZIP to the next Kaggle session, set `RESUME_ZIP` above, and rerun the notebook with the same settings. Completed jobs are skipped. Generation resumes from saved batches, while omitted activations are recomputed. Save/download outputs before session shutdown; a forcibly terminated session cannot run the export cell. You can also preserve the output directory through Kaggle's notebook-version output workflow.

The source-provenance table still marks 91 unsafe requests unresolved. To check SORRY-Bench, obtain an authorized CSV export and supply its text column to `tools/provenance.py --sources-json`; do not infer researcher authorship from a nonmatch. Read `audit/AUDIT_FINDINGS.md` for completed checks and remaining evidence gaps.


In [ ]:
from IPython.display import FileLink, display
# This cell can be rerun after a manual interruption once the preceding queue definitions exist.
export_checkpoint()
support = Path('/kaggle/working/eval_awareness_source_and_audit.zip')
with zipfile.ZipFile(support, 'w', zipfile.ZIP_DEFLATED) as z:
 for p in sorted(ROOT.rglob('*')):
 if p.is_file() and '__pycache__' not in p.parts:
 z.write(p, p.relative_to(ROOT))
display(FileLink(str(CHECKPOINT)))
display(FileLink(str(support)))
